In [1]:
from astropy.table import Table
from ligo.skymap.postprocess.crossmatch import crossmatch
import astropy_healpix as ah

/hildafs/projects/phy220048p/share/envs/desi_alerce/lib/python3.10/site-packages/ligo/lw/lsctables.py:89: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal
/hildafs/projects/phy220048p/share/envs/desi_alerce/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Read multi-order FITS skymap
skymap = Table.read("Bilby.multiorder.fits")

# Get volume-related summary info
result = crossmatch(
    skymap,
    contours=(0.5, 0.9, 0.99),   # optional
    cosmology=False        # or True for comoving volumes
)

print(result)

CrossmatchResult(searched_area=nan, searched_prob=nan, offset=nan, searched_modes=nan, contour_areas=[370.955920986403, 1681.2680876997658, 3124.4946537921446], area_probs=[], contour_modes=nan, searched_prob_dist=nan, contour_dists=[38.935833044111334, 93.85597199626625, 145.52136491242007], searched_vol=nan, searched_prob_vol=nan, contour_vols=[81326.09756283223, 473150.69036756817, 1158204.8211254033], probdensity=nan, probdensity_vol=nan)


In [3]:
v50, v90, v99 = result.contour_vols
print(f"50% volume = {v50:.3e} Mpc^3")
print(f"90% volume = {v90:.3e} Mpc^3")
print(f"99% volume = {v99:.3e} Mpc^3")

50% volume = 8.133e+04 Mpc^3
90% volume = 4.732e+05 Mpc^3
99% volume = 1.158e+06 Mpc^3


In [4]:
v99 * 0.8e-5 / 365.25 * 4

0.10147174339770816

In [5]:
v90 * 0.8e-5 / 365.25 * 4

0.04145331168175819

In [5]:
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.table import Table
from astropy.cosmology import Planck18, FlatLambdaCDM
from ligo.skymap.postprocess.crossmatch import crossmatch

skymap = Table.read("Bilby.multiorder.fits")

z = 0.03327

planck18 = Planck18
shoes = FlatLambdaCDM(H0=73.04 * u.km / u.s / u.Mpc, Om0=Planck18.Om0)

cosmologies = {
    f"Planck18 (H0={planck18.H0.value:.2f} km/s/Mpc)": planck18,
    f"SH0ES   (H0={shoes.H0.value:.2f} km/s/Mpc)":   shoes,
}

# Merger rate density: central + asymmetric errors [Mpc^-3 yr^-1]
rate     = 0.8e-5
rate_hi  = rate + 0.6e-5   # upper bound
rate_lo  = rate - 0.4e-5   # lower bound

# Search time window [yr]
dt = 4 / 365.25

for label, cosmo in cosmologies.items():
    d_L = cosmo.luminosity_distance(z)
    print(f"\n{'='*55}")
    print(f"Cosmology : {label}")
    print(f"d_L(z={z}) : {d_L:.2f}")
    print(f"{'='*55}")

    coord = SkyCoord(
        ra=175.614216 * u.deg,
        dec=34.008485 * u.deg,
        distance=d_L,
        frame="icrs",
    )

    result = crossmatch(skymap, coordinates=coord, contours=(0.5, 0.9), cosmology=cosmo)
    print(f"  2D contour level  : {result.searched_prob:.4f}")
    print(f"  Distance CDF      : {result.searched_prob_dist:.4f}")
    print(f"  3D contour level  : {result.searched_prob_vol:.4f}")
    print(f"  3D enclosed vol   : {result.searched_vol:.2f} Mpc³")
    print(f"  50% contour vol   : {result.contour_vols[0]:.2f} Mpc³")
    print(f"  90% contour vol   : {result.contour_vols[1]:.2f} Mpc³")

    # --- Chance coincidence ---
    vol = result.searched_vol          # Mpc³ inside the 3D credible volume up to the source

    N      = vol * rate    * dt
    N_hi   = vol * rate_hi * dt
    N_lo   = vol * rate_lo * dt

    print(f"\n  Chance coincidence (N expected)")
    print(f"  N = {N:.4f}  +{N_hi - N:.4f}  -{N - N_lo:.4f}")
    print(f"  (rate = {rate:.1e} +{0.6e-5:.1e} -{0.4e-5:.1e} Mpc⁻³ yr⁻¹, dt = {dt*365.25:.0f} days)")


Cosmology : Planck18 (H0=67.66 km/s/Mpc)
d_L(z=0.03327) : 151.13 Mpc
  2D contour level  : 0.7367
  Distance CDF      : 0.9699
  3D contour level  : 0.9506
  3D enclosed vol   : 611932.28 Mpc³
  50% contour vol   : 75215.26 Mpc³
  90% contour vol   : 434343.96 Mpc³

  Chance coincidence (N expected)
  N = 0.0536  +0.0402  -0.0268
  (rate = 8.0e-06 +6.0e-06 -4.0e-06 Mpc⁻³ yr⁻¹, dt = 4 days)

Cosmology : SH0ES   (H0=73.04 km/s/Mpc)
d_L(z=0.03327) : 140.00 Mpc
  2D contour level  : 0.7367
  Distance CDF      : 0.9380
  3D contour level  : 0.9052
  3D enclosed vol   : 447420.41 Mpc³
  50% contour vol   : 75215.26 Mpc³
  90% contour vol   : 434343.96 Mpc³

  Chance coincidence (N expected)
  N = 0.0392  +0.0294  -0.0196
  (rate = 8.0e-06 +6.0e-06 -4.0e-06 Mpc⁻³ yr⁻¹, dt = 4 days)


In [6]:
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.table import Table
from astropy.cosmology import Planck18, FlatLambdaCDM
from ligo.skymap.postprocess.crossmatch import crossmatch

skymap = Table.read("Bilby.multiorder.fits")
z = 0.03327
planck18 = Planck18
shoes = FlatLambdaCDM(H0=73.04 * u.km / u.s / u.Mpc, Om0=Planck18.Om0)
cosmologies = {
    f"Planck18 (H0={planck18.H0.value:.2f} km/s/Mpc)": planck18,
    f"SH0ES   (H0={shoes.H0.value:.2f} km/s/Mpc)":   shoes,
}

# Merger rate density in h^3_70 Mpc^-3 yr^-1 (as quoted by LVK)
rate_h70     = 0.8e-5
rate_h70_hi  = rate_h70 + 0.6e-5
rate_h70_lo  = rate_h70 - 0.4e-5

# Search time window [yr]
dt = 4 / 365.25

for label, cosmo in cosmologies.items():
    # h_70 = H0 / 70  =>  1 h^3_70 Mpc^-3 = h_70^3 physical Mpc^-3
    h70 = cosmo.H0.value / 70.0
    h70_3 = h70**3

    rate     = rate_h70    * h70_3
    rate_hi  = rate_h70_hi * h70_3
    rate_lo  = rate_h70_lo * h70_3

    d_L = cosmo.luminosity_distance(z)
    print(f"\n{'='*60}")
    print(f"Cosmology : {label}")
    print(f"h_70      : {h70:.4f}  (h_70^3 = {h70_3:.4f})")
    print(f"d_L(z={z}) : {d_L:.2f}")
    print(f"{'='*60}")

    coord = SkyCoord(
        ra=175.614216 * u.deg,
        dec=34.008485 * u.deg,
        distance=d_L,
        frame="icrs",
    )
    result = crossmatch(skymap, coordinates=coord, contours=(0.5, 0.9), cosmology=cosmo)
    print(f"  2D contour level  : {result.searched_prob:.4f}")
    print(f"  Distance CDF      : {result.searched_prob_dist:.4f}")
    print(f"  3D contour level  : {result.searched_prob_vol:.4f}")
    print(f"  3D enclosed vol   : {result.searched_vol:.2f} Mpc³")
    print(f"  50% contour vol   : {result.contour_vols[0]:.2f} Mpc³")
    print(f"  90% contour vol   : {result.contour_vols[1]:.2f} Mpc³")

    # --- Chance coincidence ---
    vol  = result.searched_vol
    N     = vol * rate    * dt
    N_hi  = vol * rate_hi * dt
    N_lo  = vol * rate_lo * dt
    print(f"\n  Rate (physical)   : {rate:.4e} Mpc⁻³ yr⁻¹  "
          f"[from {rate_h70:.1e} h³_70 Mpc⁻³ yr⁻¹ × h_70³={h70_3:.4f}]")
    print(f"  Chance coincidence (N expected)")
    print(f"  N = {N:.4f}  +{N_hi - N:.4f}  -{N - N_lo:.4f}")
    print(f"  (dt = {dt*365.25:.0f} days)")


Cosmology : Planck18 (H0=67.66 km/s/Mpc)
h_70      : 0.9666  (h_70^3 = 0.9030)
d_L(z=0.03327) : 151.13 Mpc
  2D contour level  : 0.7367
  Distance CDF      : 0.9699
  3D contour level  : 0.9506
  3D enclosed vol   : 611932.28 Mpc³
  50% contour vol   : 75215.26 Mpc³
  90% contour vol   : 434343.96 Mpc³

  Rate (physical)   : 7.2242e-06 Mpc⁻³ yr⁻¹  [from 8.0e-06 h³_70 Mpc⁻³ yr⁻¹ × h_70³=0.9030]
  Chance coincidence (N expected)
  N = 0.0484  +0.0363  -0.0242
  (dt = 4 days)

Cosmology : SH0ES   (H0=73.04 km/s/Mpc)
h_70      : 1.0434  (h_70^3 = 1.1360)
d_L(z=0.03327) : 140.00 Mpc
  2D contour level  : 0.7367
  Distance CDF      : 0.9380
  3D contour level  : 0.9052
  3D enclosed vol   : 447420.41 Mpc³
  50% contour vol   : 75215.26 Mpc³
  90% contour vol   : 434343.96 Mpc³

  Rate (physical)   : 9.0882e-06 Mpc⁻³ yr⁻¹  [from 8.0e-06 h³_70 Mpc⁻³ yr⁻¹ × h_70³=1.1360]
  Chance coincidence (N expected)
  N = 0.0445  +0.0334  -0.0223
  (dt = 4 days)


In [3]:
result.searched_vol * 0.8e-5 / 365.25 * 4

0.04326312436793353

In [4]:
vol = result.searched_vol
rate = 0.8e-5
rate_err_hi = 0.6e-5
rate_err_lo = 0.4e-5
dt = 4 / 365.25

N = vol * rate * dt
N_hi = vol * rate_err_hi * dt
N_lo = vol * rate_err_lo * dt

print(f"N = {N:.4f} +{N_hi:.4f} -{N_lo:.4f}")

N = 0.0433 +0.0324 -0.0216


In [15]:
# Read multi-order FITS skymap
skymap = Table.read("bayestar.multiorder.fits,2")

# Get volume-related summary info
result = crossmatch(
    skymap,
    contours=(0.5, 0.9, 0.99),   # optional
    cosmology=False        # or True for comoving volumes
)

print(result)

CrossmatchResult(searched_area=nan, searched_prob=nan, offset=nan, searched_modes=nan, contour_areas=[205.21028795543597, 786.4136995996384, 1717.9837754766586], area_probs=[], contour_modes=nan, searched_prob_dist=nan, contour_dists=[100.41645669300195, 243.3732624754048, 377.1686262556882], searched_vol=nan, searched_prob_vol=nan, contour_vols=[923388.5427309037, 4851741.063978677, 14351387.274695328], probdensity=nan, probdensity_vol=nan)


In [16]:
v50, v90, v99 = result.contour_vols
print(f"50% volume = {v50:.3e} Mpc^3")
print(f"90% volume = {v90:.3e} Mpc^3")
print(f"99% volume = {v99:.3e} Mpc^3")

50% volume = 9.234e+05 Mpc^3
90% volume = 4.852e+06 Mpc^3
99% volume = 1.435e+07 Mpc^3


In [17]:
from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.table import Table
from ligo.skymap.postprocess.crossmatch import crossmatch

# Read your skymap (multi-order FITS / MOC table)
skymap = Table.read("bayestar.multiorder.fits,2")

# 3D position of your object
coord = SkyCoord(
    ra=237.975774 * u.deg,
    dec=30.902388 * u.deg,
    distance=399.4 * u.Mpc,
    frame="icrs"
)

# You can also ask for standard contour volumes for comparison
result = crossmatch(skymap, coordinates=coord, contours=(0.5, 0.9))

print("2D contour level:", result.searched_prob)
print("distance CDF:", result.searched_prob_dist)
print("3D contour level:", result.searched_prob_vol)
print("3D enclosed volume at source:", result.searched_vol, "Mpc^3")
print("90% contour volume:", result.contour_vols[1], "Mpc^3")

2D contour level: 0.42543705725229636
distance CDF: 0.9657461988471826
3D contour level: 0.8759655152271026
3D enclosed volume at source: 4205753.522254717 Mpc^3
90% contour volume: 4851741.063978677 Mpc^3


In [18]:
result.searched_vol * 0.8e-5 / 365.25 * 4

0.36847121892443785